# Extra Theory Experiments lightweight validation

This smoke path avoids stochtree fits.

In [ ]:
# Lightweight local/Colab validation. This runs two smoke tasks only.
REPO_URL = 'https://github.com/hugogobato/DiD-BCF.git'
BRANCH = 'experiments/theory-calibration-colab'


In [ ]:
import os, sys, pathlib, subprocess, zipfile
target = pathlib.Path("DiD-BCF")
if not (target / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(target)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(target / "Extra_Theory_Experiments" / "requirements-colab.txt")], check=True)
sys.path.insert(0, str(target))
sys.path.insert(0, str(target / "Extra_Theory_Experiments" / "src"))
from extra_theory_experiments.manifest import build_manifest, manifest_frame
from extra_theory_experiments.runner import run_tasks
RESULTS_ROOT = target / "Extra_Theory_Experiments" / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
tasks = build_manifest("correction_audit", reps=2, n_shards=16, shard_id=0)[:2]
manifest_path = RESULTS_ROOT / "validation_manifest.csv"
summary_path = RESULTS_ROOT / "validation_summary.csv"
manifest_frame(tasks).to_csv(manifest_path, index=False)
summary = run_tasks(tasks, out_dir=RESULTS_ROOT / "validation_checkpoints", smoke=True, resume=True)
summary.to_csv(summary_path, index=False)
print("validated", len(summary), "rows")
output_file = str(RESULTS_ROOT / "validation_outputs.zip")
with zipfile.ZipFile(output_file, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in [manifest_path, summary_path]:
        archive.write(path, arcname=path.name)


In [ ]:
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
